<a href="https://colab.research.google.com/github/lynnlinshuo-lgtm/is4487-labs/blob/main/assignment_8_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 8: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.





## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [11]:
# Load backup Airbnb dataset from GitHub
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/airbnb_listings.csv"

df = pd.read_csv(url)

# Preview the dataset
print("Shape:", df.shape)
df.head()

Shape: (459, 77)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1.The dataset includes business information about Airbnb listings, such as listing details, host information, property characteristics, location data, room type, availability, reviews, and price-related fields. This information can help understand what factors may influence Airbnb listing prices.

2. The dataset has 459 rows and 77 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [12]:
# Drop columns that are not useful for modeling, if they exist
cols_to_drop = ['post_id', 'title', 'descr', 'details', 'address',
                'id', 'listing_url', 'name', 'description']

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Check result
print("Shape after dropping columns:", df.shape)
df.head()

Shape after dropping columns: (459, 73)


,scrape_id,last_scraped,source,neighborhood_overview,host_url,host_name,host_since,host_location,host_about,host_response_time,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,20250804133828,2025-08-04,city scrape,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,2013-01-07,"New York, NY",I am a real down to earth & cool person.,NaN,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,20250804133828,2025-08-04,city scrape,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,2014-08-07,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",within an hour,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,20250804133828,2025-08-04,city scrape,The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,2015-03-13,"Albany, NY",I am an Albany native .I have lived in Ireland...,within an hour,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,20250804133828,2025-08-04,city scrape,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,2014-08-07,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",within an hour,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,20250804133828,2025-08-04,city scrape,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,2014-07-07,"Albany, NY",NaN,NaN,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped columns such as id, listing_url, name, and description because they are identifiers, URLs, or text fields that are not directly useful for a simple regression model. These columns could add noise without helping the model predict price.

2. If these columns were included, the model might learn meaningless patterns from IDs, links, or long text instead of learning real price drivers. This could reduce accuracy, make the model harder to interpret, and potentially create biased or unreliable results.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [13]:
# Convert price to numeric if needed
if df['price'].dtype == 'object':
    df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

# Select numeric columns only
numeric_df = df.select_dtypes(include=['int64', 'float64'])

# Generate correlation matrix
corr_matrix = numeric_df.corr()

# Show variables most strongly related to price
price_corr = corr_matrix['price'].sort_values(ascending=False)

print("Correlations with price:")
print(price_corr)

Correlations with price:
price                                           1.000000
accommodates                                    0.579588
beds                                            0.547032
bedrooms                                        0.499286
bathrooms                                       0.468030
estimated_revenue_l365d                         0.249488
maximum_maximum_nights                          0.122872
minimum_maximum_nights                          0.112166
maximum_nights_avg_ntm                          0.111271
availability_30                                 0.108409
availability_60                                 0.060509
availability_90                                 0.040997
calculated_host_listings_count_entire_homes     0.033206
availability_eoy                                0.032537
review_scores_value                             0.018269
maximum_nights                                  0.017992
calculated_host_listings_count                  0.015773
review

<>:3: SyntaxWarning: invalid escape sequence '\$'
<>:3: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_5670/2556307260.py:3: SyntaxWarning: invalid escape sequence '\$'
  df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The variables with the strongest positive correlation with price were accommodates, beds, bedrooms, and bathrooms. The strongest negative correlations were review_scores_communication, longitude, review_scores_checkin, and reviews_per_month, although these negative relationships were weaker than the positive ones.


2.Useful predictors may include accommodates, beds, bedrooms, bathrooms, and estimated_revenue_l365d because they have stronger relationships with price. These variables make business sense because larger properties with more capacity and stronger revenue potential usually have higher listing prices.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [14]:
import pandas as pd

# Reload backup Airbnb dataset from GitHub
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/airbnb_listings.csv"
df = pd.read_csv(url)

# Drop columns that are not useful for modeling, if they exist
cols_to_drop = ['post_id', 'title', 'descr', 'details', 'address',
                'id', 'listing_url', 'name', 'description']

df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Convert price to numeric if needed
if df['price'].dtype == 'object':
    df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

# Define target variable
y = df['price']

# Define feature variables
X = df.drop(columns=['price'])

# Use only numeric features for now
X = X.select_dtypes(include=['int64', 'float64'])

# Drop columns with all missing values
X = X.dropna(axis=1, how='all')

# Fill remaining missing values with column median
X = X.fillna(X.median())

# Check results
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
X.head()

Feature shape: (459, 41)
Target shape: (459,)


<>:15: SyntaxWarning: invalid escape sequence '\$'
<>:15: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_5670/4191630148.py:15: SyntaxWarning: invalid escape sequence '\$'
  df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)


,scrape_id,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,20250804133828,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,28,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,20250804133828,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,20250804133828,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,2,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,20250804133828,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,20250804133828,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,1,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I am using the numeric columns in the dataset as features, excluding price because price is the target variable. These features include property size, capacity, reviews, availability, location-related numeric fields, and booking-related variables.

2. This is a regression problem because the target variable, price, is a continuous numeric value. We are predicting an amount of money, not assigning listings into categories or classes.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [15]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check split sizes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (367, 41)
X_test shape: (92, 41)
y_train shape: (367,)
y_test shape: (92,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [16]:
from sklearn.linear_model import LinearRegression

# Fit a linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict prices for the test set
y_pred = model.predict(X_test)

# Preview actual vs predicted prices
predictions = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred
})

predictions.head()

,Actual Price,Predicted Price
124,93.0,101.569487
30,54.0,94.220698
199,59.0,47.027636
438,59.0,91.578326
154,102.0,73.565446


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [17]:
from sklearn.metrics import mean_squared_error, r2_score

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Squared Error (MSE): 8752.53
R-squared (R2): -0.56


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1.The R-squared score is -0.56, which means the model does not explain the price variation well. A negative R-squared means the model performs worse than simply predicting the average price.

2. The MSE is 8752.53, which is large because the prediction errors are quite high. To improve it, I could remove weak or noisy predictors, handle outliers in price, add categorical variables through encoding, or use a more flexible model that can capture non-linear relationships.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [18]:
# Create a table of feature names and regression coefficients
coef_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

# Sort by coefficient value from highest to lowest
coef_table_sorted = coef_table.sort_values(by="Coefficient", ascending=False)

coef_table_sorted

,Feature,Coefficient
34,review_scores_value,1.351028e+02
15,minimum_nights_avg_ntm,3.650441e+01
6,bathrooms,3.263144e+01
29,review_scores_accuracy,2.087258e+01
8,beds,1.717376e+01
5,accommodates,1.654796e+01
28,review_scores_rating,1.046945e+01
17,availability_30,1.245173e+00
25,number_of_reviews_ly,3.704477e-01
23,number_of_reviews_l30d,3.177270e-01


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1.The features that increased price the most were review_scores_value, minimum_nights_avg_ntm, bathrooms, review_scores_accuracy, beds, and accommodates. These had the largest positive coefficients, meaning they were associated with higher predicted Airbnb prices.


2. Yes, some negative coefficients were surprising. For example, review_scores_communication, review_scores_checkin, review_scores_cleanliness, and bedrooms had negative coefficients. This is surprising because better reviews and more bedrooms would normally be expected to increase listing price, but the model may be affected by noise, correlations between variables, or the weak overall model fit.

3.
 A business insight is that property capacity and stay requirements appear to be important pricing factors, but the model should be interpreted carefully. Since some review-related variables behave unexpectedly, Airbnb should improve data cleaning, add categorical variables like neighborhood and room type, and test better models before using this for real pricing decisions.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [19]:
# Choose top 5 features with the strongest absolute coefficients
top_features = coef_table.copy()
top_features["AbsCoefficient"] = top_features["Coefficient"].abs()
top_5_features = top_features.sort_values(by="AbsCoefficient", ascending=False).head(5)["Feature"]

print("Top 5 features used:")
print(top_5_features.tolist())

# Create reduced training and testing sets
X_train_refined = X_train[top_5_features]
X_test_refined = X_test[top_5_features]

# Rebuild regression model using only top features
refined_model = LinearRegression()
refined_model.fit(X_train_refined, y_train)

# Predict using refined model
y_pred_refined = refined_model.predict(X_test_refined)

# Evaluate refined model
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

# Compare baseline and refined model
print(f"Baseline Model MSE: {mse:.2f}")
print(f"Baseline Model R-squared (R2): {r2:.2f}")
print(f"Refined Model MSE: {mse_refined:.2f}")
print(f"Refined Model R-squared (R2): {r2_refined:.2f}")

Top 5 features used:
['latitude', 'longitude', 'review_scores_communication', 'review_scores_value', 'review_scores_checkin']
Baseline Model MSE: 8752.53
Baseline Model R-squared (R2): -0.56
Refined Model MSE: 6401.81
Refined Model R-squared (R2): -0.14


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

✍️ Your Response: 🔧
1. I kept latitude, longitude, review_scores_communication, review_scores_value, and review_scores_checkin because they had the strongest absolute coefficients in the baseline linear regression model. These features had the largest measured impact on predicted price.

2. Yes, the model performance improved somewhat. The baseline model had an MSE of 8752.53 and an R-squared of -0.56, while the refined model had a lower MSE of 6401.81 and a better R-squared of -0.14. However, the R-squared is still negative, so the model is still not very good overall.

3. This model is simpler and easier to explain because it uses only five features instead of all numeric predictors. However, I would be careful showing it to stakeholders because the model still performs poorly and should not be used for real pricing decisions without more improvement.

4. This relates to my customized learning outcome because it shows how regression can be used to make predictions and evaluate business data. It also helps me practice interpreting model results and explaining whether a model is useful for decision-making.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


✍️ Your Response: 🔧
1. The model helped answer the business question of which Airbnb listing features may influence price and whether price can be predicted using available listing data.

2. I would recommend that Airbnb and its hosts use property and location-related factors carefully when thinking about pricing, but they should not rely only on this model. Since the model performance was still weak, it should be used as an early exploratory tool rather than a final pricing system.

3. Next, I would improve the model by using the cleaned Assignment 7 dataset, encoding categorical variables such as room type and neighborhood, removing outliers, and testing other models that can handle more complex relationships.

4. This relates to my customized learning outcome because I practiced using regression to analyze business data, evaluate model accuracy, and explain the results in a way that could support business decision-making.



## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [20]:
!jupyter nbconvert --to html "assignment_8_regression.ipynb"

[NbConvertApp] WARNING | pattern 'assignment_8_regression.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
-